In [1]:

import os
import pandas as pd
from pyspark.sql.functions import lit

raw_dir = "/lakehouse/default/Files/raw"

# Find the uploaded workbook
files = [f for f in os.listdir(raw_dir)
         if f.lower().endswith(".xlsx") and not f.startswith("~$")]

if len(files) != 1:
    raise ValueError(f"Expected one Excel file, found: {files}")

file_path = os.path.join(raw_dir, files[0])
table_name = "bronze.retail_transactions"

columns = [
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "CustomerID", "Country"
]

spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

with pd.ExcelFile(file_path, engine="openpyxl") as workbook:
    for index, sheet in enumerate(workbook.sheet_names):
        pdf = pd.read_excel(workbook, sheet_name=sheet, dtype=str)

        if len(pdf.columns) != len(columns):
            raise ValueError(f"Unexpected columns in {sheet}: {list(pdf.columns)}")

        pdf.columns = columns
        pdf = pdf.astype(object).where(pd.notna(pdf), None)

        # Record the source without modifying transaction values
        df = spark.createDataFrame(pdf)
        df = df.withColumn("SourceSheet", lit(sheet))
        df = df.withColumn("SourceFile", lit(files[0]))

        mode = "overwrite" if index == 0 else "append"

        df.write.format("delta").mode(mode).saveAsTable(table_name)

        print(f"{sheet}: {len(pdf):,} rows loaded")

        del pdf, df

print("Bronze ingestion complete.")


StatementMeta(, af6f0d6a-8155-4427-80a3-e51fa0867c12, 3, Finished, Available, Finished, False)

Year 2009-2010: 525,461 rows loaded
Year 2010-2011: 541,910 rows loaded
Bronze ingestion complete.


In [2]:
display(
    spark.sql("""
        SELECT SourceSheet, COUNT(*) AS RowCount
        FROM bronze.retail_transactions
        GROUP BY SourceSheet
        ORDER BY SourceSheet
    """)
)

StatementMeta(, af6f0d6a-8155-4427-80a3-e51fa0867c12, 4, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 1adb1cc1-c13c-44e6-88c6-db1462a1342c)

In [3]:

display(spark.sql("""
SELECT
    COUNT(*) AS TotalRows,

    SUM(CASE WHEN CustomerID IS NULL
        OR TRIM(CustomerID) = ''
        THEN 1 ELSE 0 END) AS MissingCustomerID,

    SUM(CASE WHEN Quantity IS NULL
        THEN 1 ELSE 0 END) AS MissingQuantity,

    SUM(CASE WHEN CAST(Quantity AS DOUBLE) < 0
        THEN 1 ELSE 0 END) AS NegativeQuantity,

    SUM(CASE WHEN Price IS NULL
        THEN 1 ELSE 0 END) AS MissingPrice,

    SUM(CASE WHEN CAST(Price AS DOUBLE) = 0
        THEN 1 ELSE 0 END) AS ZeroPrice,

    SUM(CASE WHEN UPPER(Invoice) LIKE 'C%'
        THEN 1 ELSE 0 END) AS CancelledInvoiceRows

FROM bronze.retail_transactions
"""))


StatementMeta(, af6f0d6a-8155-4427-80a3-e51fa0867c12, 5, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 81db3944-19d9-4fd6-b213-d126f673d546)

In [4]:

display(spark.sql("""
SELECT
    SUM(CASE
        WHEN UPPER(TRIM(Invoice)) LIKE 'C%'
         AND CAST(Quantity AS DOUBLE) < 0
        THEN 1 ELSE 0
    END) AS CancelledNegative,

    SUM(CASE
        WHEN UPPER(TRIM(Invoice)) LIKE 'C%'
         AND CAST(Quantity AS DOUBLE) >= 0
        THEN 1 ELSE 0
    END) AS CancelledNonNegative,

    SUM(CASE
        WHEN UPPER(TRIM(Invoice)) NOT LIKE 'C%'
         AND CAST(Quantity AS DOUBLE) < 0
        THEN 1 ELSE 0
    END) AS NegativeNotCancelled

FROM bronze.retail_transactions
"""))


StatementMeta(, af6f0d6a-8155-4427-80a3-e51fa0867c12, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, fe375ec5-b400-434b-ad11-9764aebb9d38)